In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# Get project directory
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

# Define file paths
RAW_PATH = BASE_DIR / "data" / "raw" / "Furniture.csv"
PROCESSED_PATH = BASE_DIR / "data" / "processed"

# Create processed folder if it doesn't exist
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Load dataset
df = pd.read_csv(RAW_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (2500, 15)


,price,cost,sales,profit_margin,inventory,discount_percentage,delivery_days,category,material,color,location,season,store_type,brand,revenue
0,218.543054,181.610932,40,16.899243,105,27.796433,9,Bed,Plastic,Red,Rural,Spring,Online,BrandA,3949.165238
1,477.821438,385.033827,7,19.418888,192,26.943715,6,Chair,Glass,Blue,Rural,Summer,Online,BrandD,-3521.002258
2,379.397274,276.736765,32,27.058842,59,21.948130,2,Table,Metal,Black,Suburban,Fall,Online,BrandD,14285.560220
3,319.396318,281.841334,48,11.758114,45,11.009944,2,Table,Glass,Green,Rural,Summer,Retail,BrandD,12261.073700
4,120.208388,69.743681,19,41.981019,35,3.183763,9,Chair,Glass,Brown,Rural,Fall,Online,BrandD,-4588.255733


In [ ]:
# Display dataset information
print("DATASET INFORMATION")
df.info()

# Check missing values
print("\nMISSING VALUES")
print(df.isnull().sum())

# Check duplicate records
print("\nDUPLICATE RECORDS")
print(df.duplicated().sum())

# Statistical summary
print("\nSTATISTICAL SUMMARY")
display(df.describe())

In [9]:
# Convert column names to lowercase
df.columns = df.columns.str.lower().str.strip()

# Remove extra spaces from text columns
for col in df.select_dtypes(include=["object", "str"]).columns:
    df[col] = df[col].str.strip()

print("Column names standardised successfully!")
print(df.columns.tolist())

Column names standardised successfully!
['price', 'cost', 'sales', 'profit_margin', 'inventory', 'discount_percentage', 'delivery_days', 'category', 'material', 'color', 'location', 'season', 'store_type', 'brand', 'revenue']


In [10]:
# Check missing values before cleaning
print("Missing values before cleaning:")
print(df.isnull().sum())

# Remove records missing essential information
df = df.dropna(
    subset=["price", "category", "material", "brand"]
)

# Fill missing numerical values with median
numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values
categorical_columns = df.select_dtypes(
    include=["object", "str"]
).columns

for col in categorical_columns:
    df[col] = df[col].fillna("Unknown")

print("\nMissing values after cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
price                  0
cost                   0
sales                  0
profit_margin          0
inventory              0
discount_percentage    0
delivery_days          0
category               0
material               0
color                  0
location               0
season                 0
store_type             0
brand                  0
revenue                0
dtype: int64

Missing values after cleaning:
price                  0
cost                   0
sales                  0
profit_margin          0
inventory              0
discount_percentage    0
delivery_days          0
category               0
material               0
color                  0
location               0
season                 0
store_type             0
brand                  0
revenue                0
dtype: int64


In [11]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Records before cleaning:", before)
print("Records after cleaning:", after)
print("Duplicates removed:", before - after)

Records before cleaning: 2500
Records after cleaning: 2500
Duplicates removed: 0


In [12]:
numeric_columns = [
    "price", "cost", "sales", "profit_margin",
    "inventory", "discount_percentage",
    "delivery_days", "revenue"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove records with invalid essential numerical values
df = df.dropna(subset=numeric_columns)

# Validate business attributes
df = df[
    (df["price"] > 0) &
    (df["cost"] >= 0) &
    (df["sales"] >= 0) &
    (df["inventory"] >= 0) &
    (df["profit_margin"].between(0, 100)) &
    (df["discount_percentage"].between(0, 100)) &
    (df["delivery_days"] >= 0)
]

print("Numerical validation completed!")
print("Remaining records:", len(df))

Numerical validation completed!
Remaining records: 2500


In [13]:
df["revenue_is_negative"] = df["revenue"] < 0

print("Negative revenue records:",
      df["revenue_is_negative"].sum())

print("\nRevenue status:")
print(df["revenue_is_negative"].value_counts())

Negative revenue records: 482

Revenue status:
revenue_is_negative
False    2018
True      482
Name: count, dtype: int64


In [14]:
df = df.reset_index(drop=True)

df.insert(
    0,
    "product_id",
    ["FUR" + str(i).zfill(5)
     for i in range(1, len(df) + 1)]
)

print("Product IDs generated successfully!")

display(df.head())

Product IDs generated successfully!


,product_id,price,cost,sales,profit_margin,inventory,discount_percentage,delivery_days,category,material,color,location,season,store_type,brand,revenue,revenue_is_negative
0,FUR00001,218.543054,181.610932,40,16.899243,105,27.796433,9,Bed,Plastic,Red,Rural,Spring,Online,BrandA,3949.165238,False
1,FUR00002,477.821438,385.033827,7,19.418888,192,26.943715,6,Chair,Glass,Blue,Rural,Summer,Online,BrandD,-3521.002258,True
2,FUR00003,379.397274,276.736765,32,27.058842,59,21.948130,2,Table,Metal,Black,Suburban,Fall,Online,BrandD,14285.560220,False
3,FUR00004,319.396318,281.841334,48,11.758114,45,11.009944,2,Table,Glass,Green,Rural,Summer,Retail,BrandD,12261.073700,False
4,FUR00005,120.208388,69.743681,19,41.981019,35,3.183763,9,Chair,Glass,Brown,Rural,Fall,Online,BrandD,-4588.255733,True


In [15]:
print("FINAL DATASET VALIDATION")

print("\nTotal records:", len(df))
print("Total columns:", len(df.columns))

print("Missing values:", df.isnull().sum().sum())

print("Duplicate records:",
      df.drop(columns=["product_id"]).duplicated().sum())

print("Unique product IDs:",
      df["product_id"].nunique())

print("Negative revenue records:",
      df["revenue_is_negative"].sum())

assert df["product_id"].is_unique
assert df.isnull().sum().sum() == 0

print("\nAll validation checks passed!")

FINAL DATASET VALIDATION

Total records: 2500
Total columns: 17
Missing values: 0
Duplicate records: 0
Unique product IDs: 2500
Negative revenue records: 482

All validation checks passed!


In [16]:
OUTPUT_PATH = PROCESSED_PATH / "products_clean.parquet"

df.to_parquet(
    OUTPUT_PATH,
    index=False,
    engine="pyarrow"
)

print("Cleaned dataset saved successfully!")
print("File location:", OUTPUT_PATH)

Cleaned dataset saved successfully!
File location: c:\Users\Nayana C\GenAI_Recommender_-FrameWorksA-\data\processed\products_clean.parquet


In [17]:
cleaned_df = pd.read_parquet(OUTPUT_PATH)

print("Parquet file loaded successfully!")
print("Dataset shape:", cleaned_df.shape)

display(cleaned_df.head())

Parquet file loaded successfully!
Dataset shape: (2500, 17)


,product_id,price,cost,sales,profit_margin,inventory,discount_percentage,delivery_days,category,material,color,location,season,store_type,brand,revenue,revenue_is_negative
0,FUR00001,218.543054,181.610932,40,16.899243,105,27.796433,9,Bed,Plastic,Red,Rural,Spring,Online,BrandA,3949.165238,False
1,FUR00002,477.821438,385.033827,7,19.418888,192,26.943715,6,Chair,Glass,Blue,Rural,Summer,Online,BrandD,-3521.002258,True
2,FUR00003,379.397274,276.736765,32,27.058842,59,21.948130,2,Table,Metal,Black,Suburban,Fall,Online,BrandD,14285.560220,False
3,FUR00004,319.396318,281.841334,48,11.758114,45,11.009944,2,Table,Glass,Green,Rural,Summer,Retail,BrandD,12261.073700,False
4,FUR00005,120.208388,69.743681,19,41.981019,35,3.183763,9,Chair,Glass,Brown,Rural,Fall,Online,BrandD,-4588.255733,True
